# Miner Tips 02: Rowmix + Q4 Rescue Baseline

This notebook turns the rowmix idea into a small, understandable experiment.

The core move is simple:

```text
for each weight row
    compare low-bit error vs q4 error
    mark the most fragile rows
    store fragile rows as q4 rescue rows
    store the rest as binary or ternary
```

This is useful for both binary and ternary submissions because pure low-bit everywhere often breaks fragile rows.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
W = rng.normal(size=(12, 16)).astype(np.float32)

# Make a few rows deliberately high-energy so they are harder to compress.
W[[2, 7, 10]] *= 4.0

print("toy weight matrix shape:", W.shape)
print("row norms:", np.round(np.linalg.norm(W, axis=1), 2))

In [ ]:
def quantize_binary_per_row(W):
    """Binary quantization with one scale per row.

    This is intentionally simple. Production recipes can use better grouping,
    learned scales, clipping, or distillation.
    """
    scale = np.mean(np.abs(W), axis=1, keepdims=True) + 1e-8
    return scale * np.sign(W)


def quantize_ternary_per_row(W):
    """Ternary quantization with one scale per row.

    Values with small magnitude become zero. The threshold here is a toy default.
    """
    threshold = 0.7 * np.mean(np.abs(W), axis=1, keepdims=True)
    mask = np.abs(W) >= threshold
    scale = np.sum(np.abs(W) * mask, axis=1, keepdims=True) / (np.sum(mask, axis=1, keepdims=True) + 1e-8)
    return scale * np.sign(W) * mask


def quantize_q4_per_row(W):
    """Toy symmetric 4-bit quantization with one scale per row."""
    qmax = 7.0
    scale = np.max(np.abs(W), axis=1, keepdims=True) / qmax + 1e-8
    q = np.clip(np.round(W / scale), -qmax, qmax)
    return q * scale


def row_mse(original, approx):
    return np.mean((original - approx) ** 2, axis=1)

In [ ]:
# Pick a base mode. Change this to "ternary" for the ternary competition.
base_mode = "binary"
low_bit = quantize_binary_per_row(W) if base_mode == "binary" else quantize_ternary_per_row(W)
q4 = quantize_q4_per_row(W)

low_bit_error = row_mse(W, low_bit)
q4_error = row_mse(W, q4)
benefit = low_bit_error - q4_error

# Rescue the top 10% most helped rows. For this tiny matrix that becomes 1 row.
rescue_fraction = 0.10
num_rescue = max(1, int(round(W.shape[0] * rescue_fraction)))
rescue_rows = np.argsort(-benefit)[:num_rescue]

mixed = low_bit.copy()
mixed[rescue_rows] = q4[rescue_rows]

print("base mode:", base_mode)
print("q4 rescue rows:", rescue_rows.tolist())
print("pure low-bit mean row mse:", round(float(np.mean(row_mse(W, low_bit))), 5))
print("mixed mean row mse:", round(float(np.mean(row_mse(W, mixed))), 5))

In [ ]:
def draw_row_plan(rescue_rows, n_rows):
    """ASCII diagram: L means low-bit row, Q means q4 rescue row."""
    symbols = []
    for row in range(n_rows):
        symbols.append("Q" if row in set(rescue_rows) else "L")
    return "rows: " + " ".join(f"{i:02d}:{s}" for i, s in enumerate(symbols))

print(draw_row_plan(rescue_rows, W.shape[0]))
print("Legend: L = binary/ternary row, Q = q4 rescue row")

## How to scale this up

For a real model, run this per module or per group:

```text
attention q/k/v/o projections
MLP gate/up/down projections
embedding and lm head
normalization modules usually stay untouched or use special handling
```

Public/dev PPL decides whether the rescue policy is good. Row MSE is only a cheap filter.